In this notebook, the files loaded into the volumes are read and stored into tables with a defined schema.

In [0]:
# Volume base path where CSV files are stored
volume_base_path = '/Volumes/Data_Lakehouse_Databricks/Bronze/Bronze_Vol/'

# Catalog and schema for bronze tables
catalog = 'Data_Lakehouse_Databricks'
schema = 'Bronze'

# Folders containing CSV files
folders = ['source_erp', 'source_crm']

print(f"Creating bronze tables in {catalog}.{schema}...\n")

for folder in folders:
    folder_path = volume_base_path + folder + '/'
    print(f"Processing folder: {folder}")
    
    try:
        # List all CSV files in the folder
        files = dbutils.fs.ls(folder_path)
        csv_files = [f for f in files if f.name.endswith('.csv')]
        
        # Process each CSV file
        for file_info in csv_files:
            # Extract filename without extension
            filename = file_info.name.replace('.csv', '')
            # Extract prefix from folder name (source_erp -> erp, source_crm -> crm)
            folder_prefix = folder.replace('source_', '')
            
            # Check if this is a streaming file (ends with _stream)
            is_streaming = filename.lower().endswith('_stream')
            
            # Remove _stream suffix for base table name if present
            base_filename = filename.lower().replace('_stream', '')
            
            # Create table name with streaming flag
            if is_streaming:
                table_name = f"bronze_{folder_prefix}_{base_filename}_stream"
            else:
                # Check if streaming version exists
                streaming_table_name = f"bronze_{folder_prefix}_{base_filename}_stream"
                streaming_full_name = f"{catalog}.{schema}.{streaming_table_name}"
                
                # Check if streaming table exists
                streaming_exists = spark.catalog.tableExists(streaming_full_name)
                
                if streaming_exists:
                    print(f"  ⚠ Skipping {file_info.name} - streaming version exists: {streaming_table_name}")
                    continue
                else:
                    table_name = f"bronze_{folder_prefix}_{base_filename}"
            
            print(f"  Reading {file_info.name}... (streaming={is_streaming})")
            
            # Read CSV with schema inference
            df = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .csv(file_info.path)
            
            # Show schema and sample data
            print(f"    Schema for {filename}:")
            df.printSchema()
            print(f"    Rows: {df.count()}")
            
            # Create fully qualified table name
            full_table_name = f"{catalog}.{schema}.{table_name}"
            
            # Write as Delta table
            df.write \
                .format("delta") \
                .mode("overwrite") \
                .saveAsTable(full_table_name)
            
            print(f"    ✓ Created table: {full_table_name}\n")
            
    except Exception as e:
        print(f"  ✗ Error processing {folder}: {str(e)}\n")

print("=== Bronze tables creation completed ===")


In [0]:
%sql
-- Query bronze_erp_cust_az12
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_erp_cust_az12 LIMIT 10

In [0]:
%sql
-- Query bronze_erp_loc_a101
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_erp_loc_a101 LIMIT 10

In [0]:
%sql
-- Query bronze_erp_px_cat_g1v2
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_erp_px_cat_g1v2 LIMIT 10

In [0]:
%sql
-- Query bronze_crm_cust_info
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_crm_cust_info LIMIT 10

In [0]:
%sql
-- Query bronze_crm_prd_info
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_crm_prd_info LIMIT 10

In [0]:
%sql
-- Query bronze_crm_sales_details
SELECT * FROM Data_Lakehouse_Databricks.Bronze.bronze_crm_sales_details LIMIT 10

### Create diagnostic table for missing info, null, ecc.

In [0]:
# Create diagnostic table for all bronze tables

catalog = 'Data_Lakehouse_Databricks'
schema = 'Bronze'

# Get all bronze tables
tables = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").filter("tableName LIKE 'bronze_%'").collect()

diagnostics = []

for table_row in tables:
    table_name = table_row['tableName']
    full_table_name = f"{catalog}.{schema}.{table_name}"
    
    print(f"Analyzing {table_name}...")
    
    try:
        # Read the table
        df = spark.table(full_table_name)
        
        # Get row count
        row_count = df.count()
        
        # Get column count
        col_count = len(df.columns)
        
        # Calculate null counts per column
        null_counts = {}
        for col in df.columns:
            null_count = df.filter(f"`{col}` IS NULL").count()
            null_counts[col] = null_count
        
        # Total null values across all columns
        total_nulls = sum(null_counts.values())
        
        # Count columns with any nulls
        cols_with_nulls = sum(1 for count in null_counts.values() if count > 0)
        
        # Calculate completeness percentage
        total_cells = row_count * col_count
        completeness_pct = ((total_cells - total_nulls) / total_cells * 100) if total_cells > 0 else 0
        
        diagnostics.append({
            'table_name': table_name,
            'row_count': row_count,
            'column_count': col_count,
            'total_null_values': total_nulls,
            'columns_with_nulls': cols_with_nulls,
            'completeness_percentage': round(completeness_pct, 2),
            'null_details': str(null_counts)
        })
        
    except Exception as e:
        print(f"  Error analyzing {table_name}: {str(e)}")
        diagnostics.append({
            'table_name': table_name,
            'row_count': None,
            'column_count': None,
            'total_null_values': None,
            'columns_with_nulls': None,
            'completeness_percentage': None,
            'null_details': f"Error: {str(e)}"
        })

# Create DataFrame from diagnostics
from pyspark.sql import Row
diagnostic_df = spark.createDataFrame([Row(**d) for d in diagnostics])

# Display the diagnostic table
print("\n=== Bronze Tables Diagnostic Summary ===")
display(diagnostic_df)

# Optionally save as a table
diagnostic_table_name = f"{catalog}.{schema}.diagnostics_bronze"
diagnostic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(diagnostic_table_name)

print(f"\n✓ Diagnostic table saved as: {diagnostic_table_name}")